# Algorithme de Deutsch-Jozsa

## Parallélisme quantique et séparation des classes de complexité

$$\text{Oracle } f: \{0,1\}^n \to \{0,1\} \quad \text{constante } \iff f(x)=c \;\forall x$$
$$\text{équilibrée } \iff f(x)=0 \text{ pour exactement } 2^{n-1} \text{ valeurs}$$

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

### Oracle constant

$$U_f |x\rangle|y\rangle = |x\rangle|y \oplus f(x)\rangle$$

Pour $f(x)=0$ (constante), $U_f = I$.

In [ ]:
def constant_oracle(n, f_val=0):
    qc = QuantumCircuit(n + 1, name='Oracle')
    if f_val == 1:
        qc.x(n)
    return qc.to_gate()

### Oracle équilibré

Pour $f(x) = x_0$ (le premier bit), l'oracle applique un CNOT du qubit 0 vers le qubit auxiliaire.

In [ ]:
def balanced_oracle(n):
    qc = QuantumCircuit(n + 1, name='Oracle')
    for i in range(n):
        qc.cx(i, n)
    return qc.to_gate()

### Circuit Deutsch-Jozsa

1. Initialiser $|0\rangle^{\otimes n}|1\rangle$
2. Hadamard sur tous les qubits
3. Oracle $U_f$
4. Hadamard sur les $n$ premiers qubits
5. Mesure

$$\text{Si résultat } = 0^{\otimes n} \implies \text{constante, sinon équilibrée}$$

In [ ]:
def deutsch_jozsa_circuit(n, oracle_type='constant', f_val=0):
    qc = QuantumCircuit(n + 1, n)
    qc.x(n)
    qc.h(range(n + 1))
    if oracle_type == 'constant':
        qc.append(constant_oracle(n, f_val), range(n + 1))
    else:
        qc.append(balanced_oracle(n), range(n + 1))
    qc.h(range(n))
    qc.measure(range(n), range(n))
    return qc

In [ ]:
n = 3
qc_const = deutsch_jozsa_circuit(n, 'constant', 0)
print(qc_const.draw())

In [ ]:
backend = AerSimulator()
result_const = backend.run(qc_const, shots=1024).result()
counts_const = result_const.get_counts()
plot_histogram(counts_const, title='Oracle constant: résultat = 000 (constante)')

In [ ]:
qc_bal = deutsch_jozsa_circuit(n, 'balanced')
result_bal = backend.run(qc_bal, shots=1024).result()
counts_bal = result_bal.get_counts()
plot_histogram(counts_bal, title='Oracle équilibré: résultat ≠ 000 (équilibrée)')

### Parallélisme quantique

L'oracle $U_f$ est appliqué à une superposition de **tous** les états d'entrée:

$$U_f\big(H^{\otimes n}|0\rangle\big)|-\rangle = \frac{1}{\sqrt{2^n}}\sum_{x=0}^{2^n-1} (-1)^{f(x)} |x\rangle |-\rangle$$

Un seul appel à l'oracle suffit : c'est l'avantage exponentiel par rapport au classique.

## Questions

**Q1.** Que se passe-t-il si on exécute l'algorithme avec un oracle constant $f(x)=1$ ? Modifier `f_val=1` et expliquer pourquoi le résultat est toujours $|00\ldots0\rangle$.

**Q2.** Généraliser l'oracle équilibré à $f(x) = x_k$ (le $k$-ième bit). Implémenter et vérifier que l'algorithme détecte toujours une fonction équilibrée.

In [ ]:
# Q1 : Oracle constant f(x)=1
qc_const1 = deutsch_jozsa_circuit(n, 'constant', 1)
counts_const1 = backend.run(qc_const1, shots=1024).result().get_counts()
plot_histogram(counts_const1, title='Oracle constant f(x)=1')

In [ ]:
# Q2 : Oracle équilibré avec f(x) = x_k
def balanced_oracle_k(n, k=0):
    qc = QuantumCircuit(n + 1, name='Oracle')
    qc.cx(k, n)
    return qc.to_gate()

def deutsch_jozsa_k(n, k):
    qc = QuantumCircuit(n + 1, n)
    qc.x(n)
    qc.h(range(n + 1))
    qc.append(balanced_oracle_k(n, k), range(n + 1))
    qc.h(range(n))
    qc.measure(range(n), range(n))
    return qc

for k in range(n):
    qc_k = deutsch_jozsa_k(n, k)
    counts_k = backend.run(qc_k, shots=1024).result().get_counts()
    print(f'k={k}: {counts_k}')